<a href="https://colab.research.google.com/github/elsa-paul11/de-portfolio-2026/blob/main/module-04-delta-lake/notebooks/m4_delta_lake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Force reinstall both together with matching versions
!pip install pyspark==3.5.3 delta-spark==3.2.1 --force-reinstall -q
print("✅ Done — now restart the runtime")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 18.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.3 which is incompatible.
✅ Done — now restart the runtime


In [1]:
!pip install pyspark==3.5.3 delta-spark==3.2.1 -q
print("✅ Installed")

✅ Installed


In [3]:
import os
import logging
import sys
from google.colab import userdata

os.environ["AWS_ACCESS_KEY_ID"]     = userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"]    = "ap-south-1"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    stream=sys.stdout
)
logger = logging.getLogger("module4")



In [4]:
# Force reconfigure logging
logger = logging.getLogger("module4")
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    ))
    logger.addHandler(handler)

logger.propagate = False  # Prevent Colab's root logger from interfering

logger.info("Logger working correctly")

2026-08-03 18:09:32,535 | INFO | Logger working correctly


In [5]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("m4_delta_lake") \
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.session.timeZone", "UTC")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
logger.info(f"Spark + Delta ready | version={spark.version}")

2026-08-03 18:09:41,735 | INFO | Spark + Delta ready | version=3.5.3


In [6]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
import random

random.seed(42)

# Simulate a customer dimension table
# 1000 customers, all starting in various cities, all GOLD tier initially
customers = []
cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Hyderabad"]

for i in range(1, 1001):
    customers.append((
        i,
        f"Customer_{i}",
        random.choice(cities),
        "GOLD",
        "2024-01-01",   # valid_from
        "9999-12-31",   # valid_to  (9999 means currently active)
        True            # is_current
    ))

schema = StructType([
    StructField("customer_id",  IntegerType(), False),
    StructField("name",         StringType(),  False),
    StructField("city",         StringType(),  True),
    StructField("tier",         StringType(),  True),
    StructField("valid_from",   StringType(),  False),
    StructField("valid_to",     StringType(),  False),
    StructField("is_current",   BooleanType(), False),
])

customers_df = spark.createDataFrame(customers, schema)

DELTA_PATH = "/tmp/delta/customers"

customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(DELTA_PATH)

logger.info(f"Delta table created | rows={customers_df.count():,}")

# Show sample
customers_df.show(5)


# Check the transaction log — the folder that makes this Delta
log_files = os.listdir(f"{DELTA_PATH}/_delta_log/")
print(f"\nTransaction log files: {sorted(log_files)}")

2026-08-03 18:09:57,065 | INFO | Delta table created | rows=1,000
+-----------+----------+---------+----+----------+----------+----------+
|customer_id|      name|     city|tier|valid_from|  valid_to|is_current|
+-----------+----------+---------+----+----------+----------+----------+
|          1|Customer_1|   Mumbai|GOLD|2024-01-01|9999-12-31|      true|
|          2|Customer_2|   Mumbai|GOLD|2024-01-01|9999-12-31|      true|
|          3|Customer_3|Bangalore|GOLD|2024-01-01|9999-12-31|      true|
|          4|Customer_4|    Delhi|GOLD|2024-01-01|9999-12-31|      true|
|          5|Customer_5|    Delhi|GOLD|2024-01-01|9999-12-31|      true|
+-----------+----------+---------+----+----------+----------+----------+
only showing top 5 rows


Transaction log files: ['.00000000000000000000.json.crc', '00000000000000000000.json', '_commits']


In [7]:
# Simulate a business event:
# 200 customers got downgraded from GOLD to SILVER in April 2024

downgrades_df = customers_df.limit(200) \
    .withColumn("tier", F.lit("SILVER"))

# Write version 2 of the table using overwrite
# (simplified — in production this would be a MERGE)
downgrades_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(DELTA_PATH)

logger.info("Version 2 written — 200 customers downgraded to SILVER")

# ── Read current version ──
print("\n=== CURRENT VERSION ===")
current_df = spark.read.format("delta").load(DELTA_PATH)
print(f"GOLD customers:   {current_df.filter(F.col('tier') == 'GOLD').count():,}")
print(f"SILVER customers: {current_df.filter(F.col('tier') == 'SILVER').count():,}")

# ── Time travel: read version 0 (before downgrades) ──
print("\n=== VERSION 0 (before downgrades) ===")
v0_df = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load(DELTA_PATH)
print(f"GOLD customers:   {v0_df.filter(F.col('tier') == 'GOLD').count():,}")
print(f"SILVER customers: {v0_df.filter(F.col('tier') == 'SILVER').count():,}")

# ── Show full version history ──
print("\n=== DELTA TABLE HISTORY ===")
deltaTable = DeltaTable.forPath(spark, DELTA_PATH)
deltaTable.history() \
    .select("version", "timestamp", "operation") \
    .show(truncate=False)

2026-08-03 18:10:31,362 | INFO | Version 2 written — 200 customers downgraded to SILVER

=== CURRENT VERSION ===
GOLD customers:   0
SILVER customers: 200

=== VERSION 0 (before downgrades) ===
GOLD customers:   1,000
SILVER customers: 0

=== DELTA TABLE HISTORY ===
+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|1      |2026-08-03 18:10:30.999|WRITE    |
|0      |2026-08-03 18:09:54.565|WRITE    |
+-------+-----------------------+---------+



In [8]:
# Start fresh with a clean table — all 1000 customers, all GOLD
# This is our "initial load" — Version 0

spark.sql(f"DROP TABLE IF EXISTS customers_delta")

customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(DELTA_PATH)

logger.info("Clean table rebuilt | all 1000 customers | all GOLD")

deltaTable = DeltaTable.forPath(spark, DELTA_PATH)
deltaTable.toDF().show(3)
deltaTable.history() \
    .select("version", "timestamp", "operation") \
    .show(truncate=False)

2026-08-03 18:10:45,190 | INFO | Clean table rebuilt | all 1000 customers | all GOLD
+-----------+----------+---------+----+----------+----------+----------+
|customer_id|      name|     city|tier|valid_from|  valid_to|is_current|
+-----------+----------+---------+----+----------+----------+----------+
|          1|Customer_1|   Mumbai|GOLD|2024-01-01|9999-12-31|      true|
|          2|Customer_2|   Mumbai|GOLD|2024-01-01|9999-12-31|      true|
|          3|Customer_3|Bangalore|GOLD|2024-01-01|9999-12-31|      true|
+-----------+----------+---------+----+----------+----------+----------+
only showing top 3 rows

+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|2      |2026-08-03 18:10:44.933|WRITE    |
|1      |2026-08-03 18:10:30.999|WRITE    |
|0      |2026-08-03 18:09:54.565|WRITE    |
+-------+-----------------------+---------+



In [9]:
# Business event:
# 50 customers moved city AND got downgraded to SILVER on 2026-07-07
# We need to:
# 1. CLOSE their old row (set valid_to, is_current=False)
# 2. INSERT a new active row (new city, SILVER tier)

# Step 1: Build the incoming changes dataset
changed_customers = []
for i in range(1, 51):
    changed_customers.append((
        i,
        f"Customer_{i}",
        "Pune",          # new city
        "SILVER",        # new tier
        "2026-07-07",   # when the change happened
    ))

updates_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name",        StringType(),  False),
    StructField("new_city",    StringType(),  True),
    StructField("new_tier",    StringType(),  True),
    StructField("change_date", StringType(),  False),
])

updates_df = spark.createDataFrame(changed_customers, updates_schema)
logger.info(f"Processing {updates_df.count()} customer changes")

# ── Step 2: CLOSE existing active rows ──
# Find the current active row for each changed customer
# and stamp it with valid_to = change_date, is_current = False
deltaTable = DeltaTable.forPath(spark, DELTA_PATH)

deltaTable.alias("target").merge(
    updates_df.alias("source"),
    # Match condition:
    # Same customer AND the row is currently active
    "target.customer_id = source.customer_id AND target.is_current = true"
).whenMatchedUpdate(set={
    "valid_to":   "source.change_date",   # close on the change date
    "is_current": F.lit(False)            # mark as historical
}).execute()

logger.info("Step 1 complete: old rows closed")

# ── Step 3: INSERT new active rows ──
# Build the new active rows to insert
new_rows_df = updates_df.select(
    F.col("customer_id"),
    F.col("name"),
    F.col("new_city").alias("city"),
    F.col("new_tier").alias("tier"),
    F.col("change_date").alias("valid_from"),
    F.lit("9999-12-31").alias("valid_to"),
    F.lit(True).alias("is_current")
)

new_rows_df.write \
    .format("delta") \
    .mode("append") \
    .save(DELTA_PATH)

logger.info("Step 2 complete: new active rows inserted")

# ── Verify: Customer 1 should now have 2 rows ──
print("\n=== Customer 1 — Full SCD Type 2 History ===")
deltaTable = DeltaTable.forPath(spark, DELTA_PATH)
deltaTable.toDF() \
    .filter(F.col("customer_id") == 1) \
    .select("customer_id", "city", "tier",
            "valid_from", "valid_to", "is_current") \
    .orderBy("valid_from") \
    .show()

# ── Verify counts ──
print("=== Table Summary ===")
all_df = deltaTable.toDF()
print(f"Total rows (including history): {all_df.count():,}")
print(f"Active rows (is_current=True):  {all_df.filter(F.col('is_current')==True).count():,}")
print(f"Historical rows:                {all_df.filter(F.col('is_current')==False).count():,}")

2026-08-03 18:10:49,649 | INFO | Processing 50 customer changes
2026-08-03 18:10:52,873 | INFO | Step 1 complete: old rows closed
2026-08-03 18:10:53,697 | INFO | Step 2 complete: new active rows inserted

=== Customer 1 — Full SCD Type 2 History ===
+-----------+------+------+----------+----------+----------+
|customer_id|  city|  tier|valid_from|  valid_to|is_current|
+-----------+------+------+----------+----------+----------+
|          1|Mumbai|  GOLD|2024-01-01|2026-07-07|     false|
|          1|  Pune|SILVER|2026-07-07|9999-12-31|      true|
+-----------+------+------+----------+----------+----------+

=== Table Summary ===
Total rows (including history): 1,050
Active rows (is_current=True):  1,000
Historical rows:                50


In [10]:
# Remember from Module 1:
# Z-ordering clusters similar values together inside Parquet files
# So Spark can skip entire files when filtering by that column
#
# Here we Z-order by city so queries like
# "WHERE city = 'Pune'" skip all non-Pune files

logger.info("Running OPTIMIZE + ZORDER BY city")

deltaTable = DeltaTable.forPath(spark, DELTA_PATH)
deltaTable.optimize().executeZOrderBy("city")

logger.info("Z-ORDER complete")

# See the full history of operations on this table
print("\n=== Full Table Operation History ===")
deltaTable.history() \
    .select("version", "timestamp", "operation") \
    .show(10, truncate=False)

2026-08-03 18:11:41,828 | INFO | Running OPTIMIZE + ZORDER BY city
2026-08-03 18:11:45,171 | INFO | Z-ORDER complete

=== Full Table Operation History ===
+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|5      |2026-08-03 18:11:44.941|OPTIMIZE |
|4      |2026-08-03 18:10:53.416|WRITE    |
|3      |2026-08-03 18:10:52.521|MERGE    |
|2      |2026-08-03 18:10:44.933|WRITE    |
|1      |2026-08-03 18:10:30.999|WRITE    |
|0      |2026-08-03 18:09:54.565|WRITE    |
+-------+-----------------------+---------+



In [11]:
# This is WHY you built SCD Type 2
# Answer: "What tier was Customer 1 in, in March 2026?"

print("=== Point-In-Time Query: Customer 1 in March 2026 ===")

deltaTable.toDF() \
    .filter(
        (F.col("customer_id") == 1) &
        (F.col("valid_from") <= "2026-03-01") &
        (F.col("valid_to")   >  "2026-03-01")
    ) \
    .select("customer_id", "city", "tier", "valid_from", "valid_to") \
    .show()

# Answer should be: Mumbai, GOLD
# Because the change to Pune/SILVER happened on 2026-07-07
# In March 2026 he was still Mumbai/GOLD

print("=== Point-In-Time Query: Customer 1 TODAY (after July 2026) ===")
deltaTable.toDF() \
    .filter(
        (F.col("customer_id") == 1) &
        (F.col("is_current") == True)
    ) \
    .select("customer_id", "city", "tier", "valid_from", "valid_to") \
    .show()

# Answer should be: Pune, SILVER

=== Point-In-Time Query: Customer 1 in March 2026 ===
+-----------+------+----+----------+----------+
|customer_id|  city|tier|valid_from|  valid_to|
+-----------+------+----+----------+----------+
|          1|Mumbai|GOLD|2024-01-01|2026-07-07|
+-----------+------+----+----------+----------+

=== Point-In-Time Query: Customer 1 TODAY (after July 2026) ===
+-----------+----+------+----------+----------+
|customer_id|city|  tier|valid_from|  valid_to|
+-----------+----+------+----------+----------+
|          1|Pune|SILVER|2026-07-07|9999-12-31|
+-----------+----+------+----------+----------+

